In [ ]:
import duckdb
from pathlib import Path


In [ ]:

ROOT = Path.cwd().parent
NOTE_DIR = ROOT / "data/physionet.org/files/mimic-iv-note/2.2/note"
PARQUET_DIR = ROOT / "data/parquet"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(ROOT / "data/mimic_note.duckdb"))


In [ ]:

for name in ["discharge", "discharge_detail", "radiology", "radiology_detail"]:
    out = PARQUET_DIR / f"{name}.parquet"
    if not out.exists():
        con.sql(f"""
            COPY (SELECT * FROM read_csv('{NOTE_DIR / name}.csv.gz', header=true))
            TO '{out}' (FORMAT parquet, COMPRESSION zstd)
        """)
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM '{out}'")


In [ ]:
con.sql("DESCRIBE discharge").show()

In [ ]:
con.sql("SELECT count(*) FROM discharge")

In [ ]:
df = con.sql("""
    SELECT *, length(text) as text_n_chars
    FROM discharge
    LIMIT 100
""").df()

In [ ]:
df.head()

In [ ]:
# print(df[df["note_id"] == "10000032-DS-21"]["text"][0])


In [ ]:
text = con.sql(
    "SELECT text FROM discharge WHERE note_id = ?",
    params=["10000032-DS-21"],
).fetchone()[0]

print(text)


In [ ]:
con.sql("DESCRIBE discharge_detail").show()

In [ ]:
con.sql("SELECT * FROM discharge_detail USING SAMPLE 1").show()
